# Canonical batch encoding

This core-only example writes a canonical `id,path,chain,model` manifest, encodes one public representation, and builds a metadata-only packed index. All generated data stays in a temporary directory.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import eris

repository = Path.cwd().resolve()
if not (repository / "tests" / "data").is_dir():
    repository = repository.parent
structure = repository / "tests" / "data" / "1AQ2.pdb"
workspace = TemporaryDirectory()
root = Path(workspace.name)
manifest = root / "structures.csv"
manifest.write_text(f"id,path,chain,model\nexample,{structure},,0\n", encoding="utf-8")
artifact_root = root / "artifacts"


In [ ]:
results = eris.batch_encode(
    manifest_path=manifest,
    representation="residue_knn_graph",
    artifact_root=artifact_root,
    results_path=root / "batch_results.csv",
    use_cache=False,
)
assert results[0].status == "success"
results[0]


In [ ]:
from eris.datasets import ErisFolderDataset

rows = eris.pack(artifact_root, root / "index.csv")
dataset = ErisFolderDataset(root / "index.csv")
assert len(dataset) == 1
assert len(rows) == 1
rows[0]


The equivalent CLI command is `eris batch encode structures.csv --representation residue_knn_graph --out artifacts`. `eris pack artifacts --out index.csv` indexes the existing Artifact v2 directories; it does not rewrite or archive them.